# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the [FAIR²](https://doi.org/10.71728/senscience.vcs2-05nj) Kilifi County Mental Health survey dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

<sub>For full schema details and field documentation, see the [SEN Science Landing Page](https://sen.science/doi/10.71728/senscience.vcs2-05nj).</sub>

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')} | License: {metadata.license}")
print(f"Authors: {len(getattr(metadata, 'author', []))}")


## 2. Data Overview
Review available record sets and their fields, referencing each by its `@id`.

> **Note:** In this dataset, the main data table is usually defined as a [record set](https://mlcommons.github.io/croissant/main/data-structure/schema/#recordset) object. For clarity and reproducibility, we always refer to all entities by their `@id`.

Let’s enumerate and inspect the available record sets and some of their field `@id`s along with field names.

In [ ]:
# List all record sets by @id
record_set_objs = list(dataset.record_sets())
record_sets = [rs['@id'] for rs in record_set_objs]
print(f"Record sets found ({len(record_sets)}):")
for i, rs_obj in enumerate(record_set_objs):
    rsid = rs_obj['@id']
    print(f" [{i}] @id: {rsid} | name: {rs_obj.get('name', '')}")
    # List a few field IDs for this record set
    fields = rs_obj.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("   Fields (@id):", [f['@id'] for f in fields][:6], ("..." if len(fields) > 6 else ""))

## 3. Data Extraction
Load data from the main survey record set into a `pandas` DataFrame for analysis. All record sets and fields are referenced via their `@id`.

Below we demonstrate loading all record sets into DataFrames, indexed by their `@id`, and inspecting the primary records table briefly.

In [ ]:
# Prepare a list of all record set @ids
record_set_ids = record_sets

dataframes = {}
for rsid in record_set_ids:
    # Each record is a dict with field @id as key
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records for record set {rsid}")

# Pick the main record set for analysis (typically the first)
main_record_set_id = record_set_ids[0]
print(f"\nMain survey record set: {main_record_set_id}\nColumns (field @id):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping on the records. Make sure to reference column names by their field `@id`.

In [ ]:
# Suppose 'gad7_total_score' (GAD-7: anxiety assessment) is a numeric field with @id, find it
main_df = dataframes[main_record_set_id]
numeric_fields = [col for col in main_df.columns if 'gad7_total_score' in col or 'phq9_total_score' in col or main_df[col].dtype.kind in 'fi']
if numeric_fields:
    # Use the first available numeric field for filtering
    numeric_field_id = numeric_fields[0]
else:
    # Otherwise pick any integer/float column
    numeric_field_id = main_df.select_dtypes(include=['number']).columns[0]

print(f"Using numeric field: {numeric_field_id}")

threshold = 10
# Filter for high scores (as an example)
filtered_df = main_df[ main_df[numeric_field_id] > threshold ]
print(f"Records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If there is a demographic field (e.g., 'demographics_gender' or similar @id), group by it
group_field_candidates = [col for col in main_df.columns if 'gender' in col or 'age_group' in col or 'education' in col]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df)
else:
    print("No obvious demographic group field for grouping found.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and breakdowns by demographic groups if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

if group_field_candidates:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id, palette='pastel')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()


## 6. Conclusion
In this notebook, you loaded the Kilifi County Mental Health Survey dataset with `mlcroissant`, explored available record sets, inspected fields using their `@id`, extracted tabular data into pandas DataFrames, and demonstrated basic data processing and visualizations using field `@id` references. 

**You are now ready to extend your analysis, further process the records, or integrate these AI-ready FAIR² standards into your ML workflows.**

> *For more details on the Croissant schema, see [mlcommons/croissant docs](https://mlcommons.org/croissant/) and field documentation in the [metadata schema](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).*